# RLVR TerminalBench Training on Colab

Train a larger model (Qwen2.5-3B-Instruct) with GRPO on a free T4 GPU (15GB VRAM).

**Runtime:** Go to Runtime > Change runtime type > Select **T4 GPU**

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q transformers trl accelerate datasets peft bitsandbytes pyyaml tqdm

In [ ]:
# Clone the repo (or upload files manually)
!git clone https://github.com/NathanG2022/rlvr-terminalbench.git 2>/dev/null || echo 'Repo already cloned'

# If repo is private or not on GitHub, upload the project files instead:
# from google.colab import files
# files.upload()  # upload a zip, then unzip

import os
os.chdir('rlvr-terminalbench')
print('Working directory:', os.getcwd())

## 2. Upload project files (if not using git clone)

If your repo isn't on GitHub, run this cell and upload a zip of the project:

In [ ]:
# Alternative: upload zip file
# from google.colab import files
# uploaded = files.upload()
# !unzip -o rlvr-terminalbench.zip
# os.chdir('rlvr-terminalbench')

## 3. Verify GPU + imports

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU found! Go to Runtime > Change runtime type > T4 GPU')

## 4. Configure model and training

In [ ]:
# Model choice - Qwen2.5-3B is free, no approval needed, 3x bigger than TinyLlama
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
# MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # smaller, for quick tests
# MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"  # needs HF approval + login

# Training params
NUM_PROMPTS = 256       # number of task prompts in dataset (should be >= 2x number of tasks)
TOTAL_STEPS = 200       # training steps (increase for better results)
BATCH_SIZE = 4          # prompts per batch
NUM_GENERATIONS = 4     # completions per prompt (more = better GRPO signal)
LEARNING_RATE = 5e-6
MAX_NEW_TOKENS = 64
MAX_STEPS_PER_EPISODE = 5

print(f'Model: {MODEL_NAME}')
print(f'Steps: {TOTAL_STEPS}, Batch: {BATCH_SIZE}, Generations: {NUM_GENERATIONS}')

## 5. Build prompt dataset

In [ ]:
import sys
sys.path.insert(0, '.')

from datasets import Dataset
from envs.terminalbench_client import TerminalBenchClient
from envs.terminalbench_env import TerminalBenchEnv

tb_client = TerminalBenchClient(command_timeout=10, max_output_length=2000)
env = TerminalBenchEnv(
    tb_client,
    max_steps=MAX_STEPS_PER_EPISODE,
    step_penalty=0.01,
    w_success=1.0,
    w_eff=0.1,
    w_quality=0.05,
)

prompts = []
for _ in range(NUM_PROMPTS):
    obs = env.reset()
    prompts.append({'prompt': obs})

train_dataset = Dataset.from_list(prompts)
print(f'Dataset: {len(train_dataset)} prompts')
print(f'\nExample prompt:\n{prompts[0]["prompt"][:300]}...')

## 6. Define reward function

In [ ]:
import re

def extract_command(text: str) -> str:
    """Extract a single shell command from model output."""
    text = text.strip()
    if not text:
        return 'echo noop'

    m = re.search(r'```(?:bash|sh)?\s*\n(.+?)```', text, re.DOTALL)
    if m:
        for line in m.group(1).splitlines():
            line = line.strip()
            if line and not line.startswith('#'):
                return line

    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        if re.match(r'^[A-Z][a-z].*\s(the|a|an|is|to|you|can|this)\s', line):
            continue
        return line

    return text.splitlines()[0].strip() or 'echo noop'


def make_reward_func():
    tb_client = TerminalBenchClient(command_timeout=10, max_output_length=2000)
    env = TerminalBenchEnv(
        tb_client,
        max_steps=MAX_STEPS_PER_EPISODE,
        step_penalty=0.01,
        w_success=1.0,
        w_eff=0.1,
        w_quality=0.05,
    )

    def reward_func(prompts, completions, **kwargs):
        rewards = []
        for prompt, completion in zip(prompts, completions):
            command = extract_command(completion)
            env.reset()
            _obs, reward, _done, _info = env.step(command)
            rewards.append(reward)
        return rewards

    return reward_func

reward_func = make_reward_func()
print('Reward function ready')

## 7. Load model with 4-bit quantization + LoRA

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
)

# Check memory usage
if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated() / 1e9
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU memory: {mem_used:.1f} / {mem_total:.1f} GB')

print('Model loaded')

## 8. Train with GRPO

In [ ]:
from trl import GRPOConfig, GRPOTrainer

OUTPUT_DIR = 'models/rlvr_colab'

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    num_generations=NUM_GENERATIONS,
    num_train_epochs=1,
    max_steps=TOTAL_STEPS,
    max_completion_length=MAX_NEW_TOKENS,
    temperature=0.7,
    top_p=0.9,
    beta=0.04,
    logging_steps=10,
    save_strategy='steps',
    save_steps=50,             # save more frequently for early stopping
    save_total_limit=5,        # keep last 5 checkpoints
    report_to='none',
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_func,
    args=grpo_config,
    train_dataset=train_dataset,
    peft_config=peft_config,
)

print(f'Training for {TOTAL_STEPS} steps...')
trainer.train()
trainer.save_model(OUTPUT_DIR)

# List saved checkpoints for best-checkpoint selection during eval
import glob
checkpoints = sorted(glob.glob(f'{OUTPUT_DIR}/checkpoint-*'))
print(f'Model saved to {OUTPUT_DIR}')
print(f'Checkpoints: {[os.path.basename(c) for c in checkpoints]}')

In [ ]:
# Restart runtime to free all GPU memory before evaluation
# After restart, skip to Section 9 below — the eval cells are self-contained
import os
os.kill(os.getpid(), 9)

## 9. Evaluate (self-contained — runs after runtime restart)

After the runtime restarts, run the cells below. They re-import everything needed.

In [ ]:
# Step 1: Evaluate TRAINED model checkpoints, pick best, save results, then restart
import os, sys, gc, json, glob, torch
from statistics import mean
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import AutoPeftModelForCausalLM

os.chdir('/content/rlvr-terminalbench')
sys.path.insert(0, '.')
from envs.terminalbench_client import TerminalBenchClient
from envs.terminalbench_env import TerminalBenchEnv

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = 'models/rlvr_colab'
MAX_NEW_TOKENS = 64
MAX_STEPS_PER_EPISODE = 5
NUM_EVAL_EPISODES = 20
NUM_QUICK_EVAL = 5  # quick eval per checkpoint for selection

def run_eval(tokenizer, model, num_episodes):
    max_ctx = getattr(model.config, 'max_position_embeddings', 2048)
    tb_client = TerminalBenchClient(command_timeout=10, max_output_length=2000)
    env = TerminalBenchEnv(
        tb_client, max_steps=MAX_STEPS_PER_EPISODE,
        step_penalty=0.01, w_success=1.0, w_eff=0.1, w_quality=0.05,
    )
    scores, steps = [], []
    for ep in range(num_episodes):
        obs = env.reset()
        done = False
        task_id = env.task.task_id if env.task else 'unknown'
        last_score = 0.0
        while not done:
            inputs = tokenizer(
                obs, return_tensors='pt', truncation=True,
                max_length=max_ctx - MAX_NEW_TOKENS,
            ).to(model.device)
            with torch.no_grad():
                out_ids = model.generate(
                    inputs['input_ids'],
                    attention_mask=inputs['attention_mask'],
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True, temperature=0.7, top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id,
                )
            new_ids = out_ids[0][inputs['input_ids'].shape[1]:]
            action = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
            action = action.split('\n')[0].strip().lstrip('$ ').strip()
            print(f'  ep{ep+1} step{env.step_count+1}: {action[:80]}')
            obs, _reward, done, info = env.step(action)
            last_score = info['success_score']
        scores.append(last_score)
        steps.append(info['step_count'])
        print(f'  -> task={task_id} score={last_score:.2f} steps={info["step_count"]}')
    return {
        'mean_score': mean(scores),
        'success_rate': mean(1.0 if s >= 1.0 else 0.0 for s in scores),
        'mean_steps': mean(steps),
    }

def load_and_eval(model_path, is_peft, num_episodes):
    quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    loader = AutoPeftModelForCausalLM if is_peft else AutoModelForCausalLM
    model = loader.from_pretrained(model_path, quantization_config=quant_config).eval()
    results = run_eval(tokenizer, model, num_episodes)
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return results

# Find all checkpoints + final model
candidates = sorted(glob.glob(f'{OUTPUT_DIR}/checkpoint-*')) + [OUTPUT_DIR]
print(f'Evaluating {len(candidates)} candidates (quick eval: {NUM_QUICK_EVAL} episodes each)...\n')

best_path, best_score = None, -1.0
for path in candidates:
    name = os.path.basename(path)
    print(f'--- {name} ---')
    results = load_and_eval(path, is_peft=True, num_episodes=NUM_QUICK_EVAL)
    print(f'  score={results["mean_score"]:.3f}')
    if results['mean_score'] > best_score:
        best_score = results['mean_score']
        best_path = path
    # Restart runtime after each to free GPU
    # Instead, we just rely on gc since quick eval is small

print(f'\n=== Best checkpoint: {os.path.basename(best_path)} (score={best_score:.3f}) ===')
print(f'Running full eval ({NUM_EVAL_EPISODES} episodes)...\n')

trained_results = load_and_eval(best_path, is_peft=True, num_episodes=NUM_EVAL_EPISODES)

with open('/content/trained_results.json', 'w') as f:
    json.dump(trained_results, f)
    json.dump({'best_checkpoint': best_path}, open('/content/best_checkpoint.json', 'w'))
print(f'\nTrained results saved: {trained_results}')

# Restart runtime to free GPU for base model eval
print('Restarting runtime to free GPU memory...')
os.kill(os.getpid(), 9)

In [ ]:
# Step 2: Re-setup after restart, then evaluate BASE model
import os, sys, gc, json, torch
from statistics import mean
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import AutoPeftModelForCausalLM

os.chdir('/content/rlvr-terminalbench')
sys.path.insert(0, '.')
from envs.terminalbench_client import TerminalBenchClient
from envs.terminalbench_env import TerminalBenchEnv

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = 'models/rlvr_colab'
MAX_NEW_TOKENS = 64
MAX_STEPS_PER_EPISODE = 5
NUM_EVAL_EPISODES = 20

def run_eval(tokenizer, model, num_episodes):
    max_ctx = getattr(model.config, 'max_position_embeddings', 2048)
    tb_client = TerminalBenchClient(command_timeout=10, max_output_length=2000)
    env = TerminalBenchEnv(
        tb_client, max_steps=MAX_STEPS_PER_EPISODE,
        step_penalty=0.01, w_success=1.0, w_eff=0.1, w_quality=0.05,
    )
    scores, steps = [], []
    for ep in range(num_episodes):
        obs = env.reset()
        done = False
        task_id = env.task.task_id if env.task else 'unknown'
        last_score = 0.0
        while not done:
            inputs = tokenizer(
                obs, return_tensors='pt', truncation=True,
                max_length=max_ctx - MAX_NEW_TOKENS,
            ).to(model.device)
            with torch.no_grad():
                out_ids = model.generate(
                    inputs['input_ids'],
                    attention_mask=inputs['attention_mask'],
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True, temperature=0.7, top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id,
                )
            new_ids = out_ids[0][inputs['input_ids'].shape[1]:]
            action = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
            action = action.split('\n')[0].strip().lstrip('$ ').strip()
            print(f'  ep{ep+1} step{env.step_count+1}: {action[:80]}')
            obs, _reward, done, info = env.step(action)
            last_score = info['success_score']
        scores.append(last_score)
        steps.append(info['step_count'])
        print(f'  -> task={task_id} score={last_score:.2f} steps={info["step_count"]}')
    return {
        'mean_score': mean(scores),
        'success_rate': mean(1.0 if s >= 1.0 else 0.0 for s in scores),
        'mean_steps': mean(steps),
    }

print(f'=== Evaluating BASE model ({MODEL_NAME}) ===')
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=quant_config).eval()
base_results = run_eval(tokenizer, base_model, NUM_EVAL_EPISODES)

with open('/content/base_results.json', 'w') as f:
    json.dump(base_results, f)
print(f'\nBase results saved: {base_results}')

In [ ]:
# Step 3: Compare results (loads from saved JSON files)
import json

with open('/content/trained_results.json') as f:
    trained_results = json.load(f)
with open('/content/base_results.json') as f:
    base_results = json.load(f)

print('='*50)
print(f'{"Metric":<25} {"Base":>10} {"Trained":>10}')
print('-'*50)
print(f'{"Mean score":<25} {base_results["mean_score"]:>10.3f} {trained_results["mean_score"]:>10.3f}')
print(f'{"Success rate":<25} {base_results["success_rate"]:>10.3f} {trained_results["success_rate"]:>10.3f}')
print(f'{"Mean steps":<25} {base_results["mean_steps"]:>10.2f} {trained_results["mean_steps"]:>10.2f}')
print('='*50)

## 10. Download trained model

In [ ]:
# Zip and download the trained model
!zip -r models/rlvr_colab.zip models/rlvr_colab/

from google.colab import files
files.download('models/rlvr_colab.zip')